In [3]:
import pyupbit
from core.upbit import Upbit
from loader.upbit_realtimedata_loader import UpbitRealtimeDataLoader
from processing.moving_average import MovingAverageProcessing
from processing.rsi import RSIProcessing
from processing.high_point_scoring import HighPointScoringProcessing
from processing.low_point_scoring import LowPointScoringProcessing
from processing.filtering_high_points import GetHighPoints
from processing.filtering_low_points import GetLowPoints
from processing.get_trend_section import  GetTrendSections
from processing.ma_200_rising import MA200RisingProcessing


from visualization.basic_price_rsi_visualization import BasicPriceWithRsiVisualization

from common.common import *

#여기가 최종 완성 구간이네

#60분일 때 
#interval_base = "minute60"
#load_count = 240
#score_band_list = [12] #12시간에 한번씩 채점
#threshold = 0.90
#indecreasing_count_set=[3,0] #3번 계속 상승구간만 체크하고 허용을 0번함

figure_to_jpg_dict = {} #저장한 이미지와 그 경로를 가진 dict 를 생성하고 아래카카오에서 이 정보를 기반으로 메시지를 전송함

#240분일 떄
interval_base = "minute240"
#interval_base = "day1"
load_count = 400
score_band_list = [30] #5일중 제일 높은 구간 채점
threshold = 0.85
indecreasing_count_set=[3,1]
ma_200_rising_min_consecutive_true = 2  # ma_200_rising 연속 True 최소 개수
aligned_in_order_window_size = 8  # 최근 8구간 모두 aligned_in_order=True 조건

target_coin_name_list = []

print("a")
## 업비트 전체 뒤져보기
tickers = pyupbit.get_tickers('KRW')

#코인별 확인
for idx, one_coin in enumerate(tickers):

    if(idx==3):
        break
    
    #if one_coin not in base_date_dict.keys(): #최근 변곡점이 아닌 애들이라면 아예 하지 않음
    #    continue
    
    if(one_coin not in ['KRW-KAITO']):
        continue
    
    print(f"{one_coin} 시작")
    
    #RAW 데이터 로드
    upbit = Upbit()
    upbit.set_loader(UpbitRealtimeDataLoader(one_coin,interval_base,load_count))
    upbit.load()
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-27 00:00:00']
    #upbit.data = upbit.data.loc[upbit.data['timestamp_kst'] <= '2025-03-26 23:00:00']
    
    #지표 추가
    indicator_list = [MovingAverageProcessing(), RSIProcessing(),HighPointScoringProcessing(score_band_list),LowPointScoringProcessing(score_band_list),MA200RisingProcessing()]
    upbit.add_sub_indicator(indicator_list)
    
    #고점, 저점 찾기
    upbit.generate_high_low_data(GetHighPoints(upbit.data.loc[upbit.data['high_score']!=0], 'high_score', threshold), 
                                GetLowPoints(upbit.data.loc[upbit.data['low_score']!=0], 'low_score', threshold)
                                )
    
    #고.저점 상승 추세구간 획득
    upbit.set_processor(GetTrendSections('high','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    upbit.set_processor(GetTrendSections('high','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    high_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.high_point_df)
    
    upbit.set_processor(GetTrendSections('low','increasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_increasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    upbit.set_processor(GetTrendSections('low','decreasing',indecreasing_count_set[0],indecreasing_count_set[1]))
    low_point_decreasing_trend_section_upbit = upbit.get_key_points(upbit.low_point_df)
    
    #저점고점 함께 상승하는 구간 찾기
    high_point_group_start_end_upbit = []
    for i in high_point_increasing_trend_section_upbit:
        high_point_group_start_end_upbit.append([i[0],i[-1]])
    
    
    low_point_group_start_end_upbit = []
    for i in low_point_increasing_trend_section_upbit:
        low_point_group_start_end_upbit.append([i[0],i[-1]])

    
    #고점은 낮아지는 구간 찾기
    #high_point_group_start_end_upbit = []
    #for i in high_point_decreasing_trend_section_upbit:
    #    high_point_group_start_end_upbit.append([i[0],i[-1]])
    
    
    
    high_low_increasing_trend_section_upbit, high_low_list_dict = find_overlapping_intervals(high_point_group_start_end_upbit, low_point_group_start_end_upbit)
    
    #중첩되었을 떄의 고점 리스트는
    
    #중첩 구간이 가장 최근인지 확인
    if(len(high_low_increasing_trend_section_upbit) != 0 ):
        #중첩된 구간의 마지막 인덱스를 찾고
        latest_overlap_index = high_low_increasing_trend_section_upbit[-1][-1] #가장 가까운 구간의 인덱스 값을 찾고
        #중첩이 되었을 떄 활용되었던 고점의 마지막 인덱스를 획득하고
        latest_high_index = high_low_list_dict[latest_overlap_index]['high_section'][-1]
        
        #중첩이 되었을 때 활용되었던 저점의 마지막 인덱스를 획득
        latest_low_index = high_low_list_dict[latest_overlap_index]['low_section'][-1]
        
        print(f"기준이 되는 고점 날짜 ; {upbit.data.iloc[[latest_high_index]]['timestamp_kst'].iloc[0]}")
        print(f"기준이 되는 저점 날짜 ; {upbit.data.iloc[[latest_low_index]]['timestamp_kst'].iloc[0]}")
        
        latest_timestamp_high = upbit.data.iloc[[latest_high_index]]['timestamp_kst'].iloc[0] #마지막 고점구간의 끝 날짜 확인
        latest_timestamp_low = upbit.data.iloc[[latest_low_index]]['timestamp_kst'].iloc[0] #마지막 저점구간의 끝 날짜 확인
        check_df_high = upbit.data.loc[upbit.data['timestamp_kst'] >= latest_timestamp_high] #그 날짜보다 높은 값이 있는지 확인
        check_df_low = upbit.data.loc[upbit.data['timestamp_kst'] >= latest_timestamp_low] #그 날짜보다 높은 값이 있는지 확인
        
        print(f"기준 고점 이후 몇시간 지났나 : {len(check_df_high)}")
        print(f"기준 저점 이후 몇시간 지났나 : {len(check_df_low)}")
        
        #print(check_df)
        #print(len(check_df))
        if(len(check_df_high)<11 or len(check_df_low)<11): #1개만 발견됐다면 이건 바로 지금 유의미한 자리인 것
            pass
        else:
            continue
    else: #중첩 구간이 없는건 일단 무시
        continue
    
    
    if('aligned_in_order' not in upbit.data.columns or 'ma_200_rising' not in upbit.data.columns):
        continue

    # 1) ma_200_rising 이 2번 연속 True 인 지점
    ma_rising_series = upbit.data['ma_200_rising'].fillna(False).astype(bool)
    ma_rising_two_consecutive = ma_rising_series
    for shift_step in range(1, ma_200_rising_min_consecutive_true):
        ma_rising_two_consecutive = ma_rising_two_consecutive & ma_rising_series.shift(shift_step, fill_value=False)

    # 2) 최근 8구간의 aligned_in_order 가 모두 True 인 지점
    aligned_series = upbit.data['aligned_in_order'].fillna(False).astype(bool)
    aligned_all_true_in_8 = aligned_series.rolling(
        window=aligned_in_order_window_size,
        min_periods=aligned_in_order_window_size
    ).sum().eq(aligned_in_order_window_size)

    # 두 조건을 동시에 만족하는 지점을 진한 노란색 후보로 사용
    dark_yellow_mask = ma_rising_two_consecutive & aligned_all_true_in_8

    if(not dark_yellow_mask.any()):
        continue

    # add_vrect_to_main_figure 에 주입할 [시작인덱스, 끝인덱스] 형태 구간 생성
    dark_yellow_group_id = (dark_yellow_mask != dark_yellow_mask.shift(fill_value=False)).cumsum()
    dark_yellow_group_start_end_upbit = []
    for _, one_group_df in upbit.data.loc[dark_yellow_mask].groupby(dark_yellow_group_id[dark_yellow_mask]):
        dark_yellow_group_start_end_upbit.append([one_group_df.index[0], one_group_df.index[-1]])
    
    #일단 주목해야할 코인들의 리스트를 받은
    target_coin_name_list.append(one_coin)
    
    
    #출력
    draw_instance = BasicPriceWithRsiVisualization()
    
    draw_instance.set_data(upbit.data, upbit.high_point_df, upbit.low_point_df)
    draw_instance.make_figure()
    
    figure = draw_instance.get_figure()
    
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_low_increasing_trend_section_upbit) #중첩된 구간만 그리기
    #add_vrect_to_main_figure(figure, upbit.data, 'red', high_point_group_start_end_upbit) #고점 그래프는 빨간색
    #add_vrect_to_main_figure(figure, upbit.data, 'blue', low_point_group_start_end_upbit) #고점 그래프는 파란색
    add_vrect_to_main_figure(figure, upbit.data, '#b8860b', dark_yellow_group_start_end_upbit) #조건 충족 구간은 진한 노란색
    draw_instance.visualize()
    
    #파일이름으로 저장함
    
    #jpg_file_name = JPG_DIRECTORY+'/'+generate_jpg_file_name(one_coin)
    #print(f"파일 저장 :{jpg_file_name}")
    #figure.write_image(jpg_file_name, format="jpg")
    #print("완료")
    
    
    #break
    
    
    
    
    


a


In [1]:
import pyupbit

In [3]:
import pandas as pd

import pandas as pd

pd.set_option("display.max_rows", None)         # 모든 행
pd.set_option("display.max_columns", None)      # 모든 열
pd.set_option("display.width", None)            # 줄바꿈 폭 제한 해제
pd.set_option("display.max_colwidth", None)     # 컬럼 내용 생략(...) 방지
pd.set_option("display.expand_frame_repr", False)  # 가로로 펼쳐서 보기
upbit.data[pd.to_datetime(upbit.data["timestamp_kst"]).dt.strftime("%Y-%m") == "2026-03"]

,timestamp_kst,open,low,high,close,ma_20,ma_60,ma_200,aligned_in_order,RSI,high_score,low_score,ma_200_rising
258,2026-03-01 01:00:00,485.0,485.0,509.0,497.0,479.30,454.133333,491.880,False,62.393162,14,0,False
259,2026-03-01 05:00:00,497.0,495.0,505.0,497.0,480.45,454.900000,491.365,False,62.931034,0,0,False
260,2026-03-01 09:00:00,498.0,489.0,508.0,500.0,481.40,455.600000,490.870,False,64.406780,0,0,False
261,2026-03-01 13:00:00,500.0,493.0,501.0,496.0,482.35,456.366667,490.335,False,61.016949,0,0,False
262,2026-03-01 17:00:00,496.0,487.0,498.0,487.0,482.90,457.066667,489.800,False,53.781513,0,0,False
263,2026-03-01 21:00:00,489.0,482.0,495.0,488.0,484.00,457.950000,489.240,False,50.892857,0,0,False
264,2026-03-02 01:00:00,488.0,480.0,489.0,480.0,484.60,458.733333,488.630,False,45.217391,0,0,False
265,2026-03-02 05:00:00,480.0,473.0,484.0,478.0,485.15,459.350000,488.035,False,39.252336,0,0,False
266,2026-03-02 09:00:00,478.0,477.0,492.0,489.0,486.30,460.050000,487.420,False,48.181818,0,0,False
267,2026-03-02 13:00:00,489.0,472.0,490.0,481.0,486.85,460.716667,486.695,False,42.982456,0,10,False
